# Event Detection

Per-game step: detect passes, shots, and possession changes from player + ball trajectories on the pitch.

**Status:** Scaffold — blocked on reliable homography for Tier 2/3 matches.

## Inputs (from upstream steps)
- Per-frame detections with track IDs + team labels → `01_team_classification.ipynb`
- Per-frame 3×4 projection matrix → `02_homography.ipynb`
- Period boundaries (first/second half frame indices) → `data/period_detection_results.json`

## Data flow
```
Frame
  → YOLO detect players + ball                  (PlayerDetector)
  → BoT-SORT track                               (Tracker)
  → ResNet18 embed → GMM/supervised classify     (TeamClassifier)
  → PnLCalib project → pitch (x, y) metres       (run_pnlcalib_video)

Per-frame pitch state:
  players: [(track_id, team, x, y), ...]
  ball:    (x, y) or None

Event detection (across frames):
  ball carrier  = player nearest ball within CARRY_THRESHOLD
  possession    = team of ball carrier
  pass          = ball leaves carrier, arrives at teammate
  shot          = ball moves toward goal, no same-team receiver
  poss. change  = carrier team switches (debounced)
```

## Output schema
```python
@dataclass
class Event:
    frame:        int    # frame index in source video
    time_sec:     float  # match clock time (seconds from kickoff)
    event_type:   str    # 'pass' | 'shot' | 'possession_change'
    team:         int    # 0 = Team A, 1 = Team B
    player_track: int    # track_id of initiating player
    x:            float  # pitch x (metres, 0–105)
    y:            float  # pitch y (metres, 0–68)
```

In [ ]:
import sys
import json
import cv2
import numpy as np
import matplotlib.pyplot as plt
import importlib
from pathlib import Path
from dataclasses import dataclass, asdict

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import src.config, src.detection, src.tracking, src.team_classifier, src.video_utils
for mod in [src.config, src.detection, src.tracking, src.team_classifier, src.video_utils]:
    importlib.reload(mod)

from src.config import Config
from src.detection import PlayerDetector, BallInterpolator
from src.tracking import Tracker
from src.team_classifier import TeamClassifier
from src.video_utils import open_video, get_video_info

print('Loaded.')

In [ ]:
GAME_SLUG = 'bud-sut'  # <- change this per game

video_path = Config.MATCH_VIDEOS[GAME_SLUG]
periods    = json.loads((PROJECT_ROOT / 'data' / 'period_detection_results.json').read_text())
period     = next(r for r in periods if r['slug'] == GAME_SLUG)

fps      = period['fps']
fh_start = period['first_half_start_frame']
fh_end   = period['first_half_end_frame']
sh_start = period['second_half_start_frame']

print(f'{GAME_SLUG}: {video_path.name}')
print(f'First half:  frame {fh_start} → {fh_end} ({(fh_end-fh_start)/fps/60:.1f} min)')
print(f'Second half: frame {sh_start} → end')

## Step 1: Run Detection + Tracking + Team Classification

> For the full match this can take 10–30 min. Run once and cache the output.

In [ ]:
# Load the saved classifier for this game
clf_path = Config.OUTPUT_CLASSIFIERS_DIR / f'{GAME_SLUG}_classifier.pkl'
if not clf_path.exists():
    raise FileNotFoundError(f'No classifier for {GAME_SLUG}. Run 01_team_classification.ipynb first.')

game_clf = TeamClassifier.load(clf_path)

# TODO: run detection + tracking on the full first half, apply game_clf per detection.
# Output: all_detections list (per-frame dicts with track_id + team_id).
#
# For the full 45-minute half at 25fps = ~67,500 frames — expect ~45 min processing time.
# Consider saving the detection output to a cache file to avoid re-running.
#
# all_detections = run_full_half(video_path, fh_start, fh_end, game_clf)  # TODO

## Step 2: Load Homography

> Load per-frame projection matrices from `02_homography.ipynb`.

In [ ]:
# TODO: load per-frame P matrices (3x4) from homography output.
# When P is None, fall back to last known-good projection.
#
# projections[frame_idx] = P (np.ndarray 3x4) or None

# projections = load_cached_projections(GAME_SLUG)  # TODO


def feet_to_pitch(bbox, P):
    '''Project bottom-centre of a bounding box to pitch coordinates (metres).'''
    x1, y1, x2, y2 = bbox
    foot = np.array([(x1 + x2) / 2, y2, 1.0])
    H    = np.linalg.inv(P[:, [0, 1, 3]])
    p    = H @ foot
    if abs(p[2]) < 1e-10:
        return None
    return p[0] / p[2] + 52.5, p[1] / p[2] + 34.0

## Step 3: Build Pitch Trajectories

For each frame, project every player's foot position to pitch coordinates.

In [ ]:
# TODO: build per-track trajectory
#
# track_positions[track_id] = [
#     {'frame': int, 'x': float, 'y': float, 'team': int},
#     ...
# ]
#
# ball_positions[frame_idx] = (x, y) or None

## Step 4: Pass Detection

**Logic:**
1. Ball carrier = player nearest to ball within `CARRY_THRESHOLD` metres
2. Pass starts when ball leaves carrier's radius and moves toward a teammate
3. Pass completes when a teammate comes within `RECEIVE_THRESHOLD` metres
4. Record: `(frame, team, from_track, to_track, origin_x, origin_y, dest_x, dest_y)`

In [ ]:
CARRY_THRESHOLD_M   = 2.0   # metres — ball is 'carried' if player within this distance
RECEIVE_THRESHOLD_M = 2.0   # metres — pass received when teammate comes this close

# TODO: implement pass detection
# passes = detect_passes(ball_positions, track_positions, CARRY_THRESHOLD_M, RECEIVE_THRESHOLD_M)

## Step 5: Shot Detection

**Logic:**
1. Ball moves toward goal end (x > `SHOT_X_THRESHOLD` or x < 105 - `SHOT_X_THRESHOLD`)
2. Ball not received by a same-team player within `RECEIVE_THRESHOLD` metres
3. Record: `(frame, team, player_track, origin_x, origin_y)`

In [ ]:
SHOT_X_THRESHOLD_M = 78.0   # metres from left goal line (78m = roughly edge of penalty arc)
GOAL_Y_MIN_M       = 22.32  # goal post y positions
GOAL_Y_MAX_M       = 45.68

# TODO: implement shot detection
# shots = detect_shots(ball_positions, track_positions, SHOT_X_THRESHOLD_M)

## Step 6: Possession Changes

**Logic:**  
Possession changes when the ball carrier switches team. Debounce with `MIN_POSSESSION_FRAMES` to suppress noise from misclassified frames or brief interceptions.

In [ ]:
MIN_POSSESSION_FRAMES = 5   # possession must hold for this many frames to count as a change

# TODO: implement possession tracking
# possession_changes = detect_possession_changes(ball_positions, track_positions, MIN_POSSESSION_FRAMES)

## Step 7: Export

Write all events to `output/events/{slug}_events.json`.

In [ ]:
@dataclass
class Event:
    frame:        int
    time_sec:     float
    event_type:   str    # 'pass' | 'shot' | 'possession_change'
    team:         int    # 0 = Team A, 1 = Team B
    player_track: int
    x:            float  # pitch metres, 0–105
    y:            float  # pitch metres, 0–68


def export_events(events, slug):
    Config.OUTPUT_EVENTS_DIR.mkdir(parents=True, exist_ok=True)
    out_path = Config.OUTPUT_EVENTS_DIR / f'{slug}_events.json'
    with open(out_path, 'w') as f:
        json.dump([asdict(e) for e in events], f, indent=2)
    print(f'Saved {len(events)} events to {out_path}')


# TODO: combine passes + shots + possession_changes into one list and call export_events